# Setup & Initialization

In [1]:
import pyspark
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F
import ConnectionConfig as cc
from importlib import reload

reload(cc)

cc.setupEnvironment()
cc.listEnvironment()

Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/spark/conf
CONDA_DEFAULT_ENV: base
NB_USER: jovyan
LC_ALL: C

In [2]:
spark = cc.startLocalCluster("dimHour")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


# Creating Hour dimension from scratch

In [8]:
# Generate hours 0 to 23
df_hour = spark.range(0, 24).withColumnRenamed("id", "hour")

# Add more columns
# df_dim_hour = df_hour.withColumn("hour_label", F.format_string("%02d:00", F.col("hour"))) \
#     .withColumn("time_of_day",
#                 F.when((F.col("hour") >= 5) & (F.col("hour") < 12), "Morning")
#                 .when((F.col("hour") >= 12) & (F.col("hour") < 17), "Afternoon")
#                 .when((F.col("hour") >= 17) & (F.col("hour") < 21), "Evening")
#                 .otherwise("Night")) \
#     .withColumn("hour_sk", F.col("hour")) # Using hour as SK since it is unique and fixed


df_dim_hour = df_hour.withColumn("hour_sk", F.col("hour")) # Using hour as SK since it is unique and fixed


df_dim_hour.show(24)

+----+-------+
|hour|hour_sk|
+----+-------+
|   0|      0|
|   1|      1|
|   2|      2|
|   3|      3|
|   4|      4|
|   5|      5|
|   6|      6|
|   7|      7|
|   8|      8|
|   9|      9|
|  10|     10|
|  11|     11|
|  12|     12|
|  13|     13|
|  14|     14|
|  15|     15|
|  16|     16|
|  17|     17|
|  18|     18|
|  19|     19|
|  20|     20|
|  21|     21|
|  22|     22|
|  23|     23|
+----+-------+



# Writing the data to a delta-file

In [ ]:
# Delete delta table if it already exists
cc.delete_delta_table(spark, "dim_hour")

In [9]:
df_dim_hour.write.format("delta").mode("overwrite").save("spark-warehouse/dim_hour")

In [10]:
# Verify
spark.read.format("delta").load("spark-warehouse/dim_hour").show()

+----+-------+
|hour|hour_sk|
+----+-------+
|   0|      0|
|   1|      1|
|   2|      2|
|   3|      3|
|   4|      4|
|   5|      5|
|   6|      6|
|   7|      7|
|   8|      8|
|   9|      9|
|  10|     10|
|  11|     11|
|  12|     12|
|  13|     13|
|  14|     14|
|  15|     15|
|  16|     16|
|  17|     17|
|  18|     18|
|  19|     19|
+----+-------+
only showing top 20 rows


## Save as a Parquet format

In [ ]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/dim_hour_parquet")

In [11]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_hour")

df.coalesce(1).write.format("parquet").save("./parquet/dim_hour_parquet")

In [12]:
spark.stop()
%reset